# Network Intrusion Detection: Rediscovering Denning's Statistical Anomaly Law

## **Table of Contents**

- [1 - Business Objective](#1-business-objective)
  - [1.1 - Overview](#11-overview)
  - [1.2 - Business Objective Statement](#12-business-objective-statement)
- [2 - Problem Statement](#2-problem-statement)
  - [2.1 - Overview](#21-overview)
- [3 - Solution Methodology](#3-solution-methodology)
  - [3.1 - Overview](#31-overview)
  - [3.2 - The Rediscovery Standard](#32-the-rediscovery-standard)
- [4 - A Brief History of Intrusion Detection](#4-a-brief-history-of-intrusion-detection)
  - [4.1 - Overview](#41-overview)
  - [4.2 - Milestones in Intrusion Detection](#42-milestones-in-intrusion-detection)
- [5 - The Science: Statistical Deviation Detection and Network Traffic](#5-the-science-statistical-deviation-detection-and-network-traffic)
  - [5.1 - Denning's Statistical Anomaly Model](#51-dennings-statistical-anomaly-model)
  - [5.2 - Flow-Level Feature Definitions](#52-flow-level-feature-definitions)
  - [5.3 - Self-Similarity and Heavy-Tailed Traffic](#53-self-similarity-and-heavy-tailed-traffic)
  - [5.4 - How the Three Attack Types Deviate from Baseline](#54-how-the-three-attack-types-deviate-from-baseline)
- [6 - Installing and Importing the Libraries](#6-installing-and-importing-the-libraries)
  - [6.1 - Overview](#61-overview)
- [7 - Generating the Synthetic Network Flow Dataset](#7-generating-the-synthetic-network-flow-dataset)
  - [7.1 - Overview](#71-overview)
  - [7.2 - Benign Traffic Generation](#72-benign-traffic-generation)
  - [7.3 - Attack Traffic Generation](#73-attack-traffic-generation)
  - [7.4 - Class Balance Check](#74-class-balance-check)
- [8 - Exploratory Data Analysis and Human-Engineered Features](#8-exploratory-data-analysis-and-human-engineered-features)
  - [8.1 - Overview](#81-overview)
  - [8.2 - Human-Engineered Feature 1 and 2: SYN Ratio and RST Ratio](#82-human-engineered-feature-1-and-2-syn-ratio-and-rst-ratio)
  - [8.3 - Human-Engineered Feature 3 and 4: Packet Rate and Byte Rate](#83-human-engineered-feature-3-and-4-packet-rate-and-byte-rate)
  - [8.4 - Human-Engineered Feature 5 and 6: Byte-to-Packet Ratio and Port Fan-Out Rate](#84-human-engineered-feature-5-and-6-byte-to-packet-ratio-and-port-fan-out-rate)
  - [8.5 - Human-Engineered Feature 7 and 8: Connection Repetition and Timing Regularity](#85-human-engineered-feature-7-and-8-connection-repetition-and-timing-regularity)
  - [8.6 - Human-Engineered Feature 9, 10, and 11: Deviation from Source Baseline](#86-human-engineered-feature-9-10-and-11-deviation-from-source-baseline)
- [9 - GenAI-Generated Features](#9-genai-generated-features)
  - [9.1 - Overview](#91-overview)
  - [9.2 - Constructing the AI-Suggested Features](#92-constructing-the-ai-suggested-features)
  - [9.3 - Ablation: Human Features vs. AI Features vs. Combined](#93-ablation-human-features-vs-ai-features-vs-combined)
- [10 - GenAI Synthetic Data Augmentation](#10-genai-synthetic-data-augmentation)
  - [10.1 - Overview](#101-overview)
  - [10.2 - Sampling Synthetic Rows from the LLM-Proposed Parameters](#102-sampling-synthetic-rows-from-the-llm-proposed-parameters)
  - [10.3 - Three-Way Generalization Check](#103-three-way-generalization-check)
- [11 - Classical ML Model: Gradient Boosting and Random Forest](#11-classical-ml-model-gradient-boosting-and-random-forest)
  - [11.1 - Overview](#111-overview)
  - [11.2 - Selecting the Reference Classical Model](#112-selecting-the-reference-classical-model)
- [12 - Light Deep Learning Model: PyTorch Feedforward Network](#12-light-deep-learning-model-pytorch-feedforward-network)
  - [12.1 - Overview](#121-overview)
  - [12.2 - Comparing the Neural Network to the Classical Model](#122-comparing-the-neural-network-to-the-classical-model)
- [13 - Foundation Model Benchmark: TabPFN](#13-foundation-model-benchmark-tabpfn)
  - [13.1 - Overview](#131-overview)
  - [13.2 - Fitting TabPFN on a Subsampled Training Set](#132-fitting-tabpfn-on-a-subsampled-training-set)
  - [13.3 - Three-Way Model Comparison](#133-three-way-model-comparison)
- [14 - Explainability: SHAP](#14-explainability-shap)
  - [14.1 - Why SHAP for This Case](#141-why-shap-for-this-case)
  - [14.2 - Global Feature Importance](#142-global-feature-importance)
  - [14.3 - Local Explanation for a Flagged Flow](#143-local-explanation-for-a-flagged-flow)
- [15 - The Law Rediscovery Moment: Statistical Deviation Over Signatures](#15-the-law-rediscovery-moment-statistical-deviation-over-signatures)
  - [15.1 - What the Explanation Surfaces](#151-what-the-explanation-surfaces)
  - [15.2 - A Deviation Score Separates All Three Attack Types](#152-a-deviation-score-separates-all-three-attack-types)
  - [15.3 - The Historical Payoff](#153-the-historical-payoff)
- [16 - Agentic Layer: LangGraph Security Recommendation](#16-agentic-layer-langgraph-security-recommendation)
  - [16.1 - Overview](#161-overview)
  - [16.2 - Running the Agent on the Flagged Flow](#162-running-the-agent-on-the-flagged-flow)
  - [16.3 - From Fixed Pipeline to Autonomous Agent](#163-from-fixed-pipeline-to-autonomous-agent)
  - [16.4 - Building the ReAct Tool-Calling Agent](#164-building-the-react-tool-calling-agent)
  - [16.5 - Running the Autonomous Agent on the Flagged Flow](#165-running-the-autonomous-agent-on-the-flagged-flow)
- [17 - Interactive Prediction Demo](#17-interactive-prediction-demo)
  - [17.1 - Overview](#171-overview)
  - [17.2 - Calling the Autonomous Agent on a Demo Flow](#172-calling-the-autonomous-agent-on-a-demo-flow)
- [18 - Conclusion and Takeaways](#18-conclusion-and-takeaways)
  - [18.1 - Conclusion](#181-conclusion)
  - [18.2 - Takeaways](#182-takeaways)

## **1 - Business Objective**

### **1.1 - Overview**

A network intrusion detection system inspects traffic and decides, flow by flow, whether that
traffic reflects normal use of a network or an attempt to compromise it. A security operations team
that must review every alert by hand cannot keep pace with modern traffic volumes, and a system that
misses attacks or buries analysts in false positives is worse than no system at all.

The cost of getting this wrong is well documented. Denial-of-service floods take down services that
customers depend on. Port scans are the reconnaissance phase of nearly every larger intrusion,
carried out before an attacker knows which service to target. Brute-force login attempts against
exposed management ports (SSH, RDP) remain one of the most common ways an external attacker gains an
initial foothold on a network, precisely because they require no software vulnerability, only enough
attempts.

The business objective of this case study is to build a model that classifies network flows as
benign or malicious from flow-level telemetry alone, features a network monitoring appliance can
compute without inspecting packet payloads. The secondary objective is explainability: the model must
expose which telemetry features actually drive a malicious classification, so that a security team can
trust an alert and act on it, rather than treating the classifier as an opaque box.

### **1.2 - Business Objective Statement**

Given flow-level telemetry for a network connection (duration, packet and byte counts, TCP flag
counts, destination-port fan-out for the source, inter-arrival timing statistics, and a per-source
behavioral baseline), classify the flow as benign or malicious, and identify which telemetry features
the model relies on most so that a security analyst can check the result against established
intrusion-detection theory.

## **2 - Problem Statement**

### **2.1 - Overview**

This is a supervised binary classification problem. The target variable is a benign or malicious
label. A secondary multiclass label (attack type) is carried alongside the binary target for analysis,
but the primary modeling task is binary. The input variables are:

| Variable | Description |
|---|---|
| flow_duration_s | Duration of the flow in seconds |
| packet_count | Total packets observed in the flow |
| byte_count | Total bytes observed in the flow |
| syn_count, ack_count, fin_count, rst_count | TCP flag counts within the flow |
| dst_port | Destination port targeted by the flow |
| unique_dst_ports_window | Distinct destination ports contacted by this source in the observation window |
| src_port_entropy_window | Entropy of source ports used by this source in the observation window |
| flows_per_src_dst_pair_window | Count of flows from this source to this exact destination and port in the window |
| inter_arrival_mean_ms, inter_arrival_cv | Mean and coefficient of variation of inter-arrival time between packets |
| baseline_pkt_rate_src, baseline_byte_rate_src, baseline_unique_ports_src | This source's established historical baseline for packet rate, byte rate, and destination-port fan-out |

No public dataset exposes raw flow telemetry at this granularity alongside a clean ground-truth label
for exactly these four traffic classes, so this case study generates a synthetic dataset from
statistical models of benign and attack traffic (Section 7), calibrated against the qualitative
signatures reported in the network security and measurement literature (Section 5).

The evaluation metric is F1 score and ROC-AUC on the held-out test split, since malicious flows are a
minority class and accuracy alone would reward a model that simply predicts benign for everything. A
security team would consider a model useful for automated triage if it recovers the large majority of
malicious flows at a false-positive rate low enough that analysts are not overwhelmed.

## **3 - Solution Methodology**

### **3.1 - Overview**

The notebook proceeds in stages. First, a synthetic dataset is generated from statistical models of
benign traffic and three attack types (SYN-flood denial of service, port scanning, and brute-force
login attempts). Exploratory analysis follows, alongside a set of features a network security analyst
would compute by hand: SYN and RST ratios, packet and byte rate, byte-to-packet ratio, destination-port
fan-out, connection repetition, timing regularity, and deviation from each source's established
baseline.

A small open-weight language model is then called through Hugging Face's free inference router to
propose a second, independent set of candidate features from the raw schema. An ablation compares a
gradient boosting model trained on the human-engineered features alone, the AI-suggested features
alone, and the combined set, to test whether the two feature sources are complementary.

The same language model is then used to propose realistic parameter distributions for
under-represented regions of the traffic space (slow, low-rate port scans and large benign transfers).
Synthetic rows are sampled from those distributions using the same generation logic as the original
data, and a three-way comparison checks that models trained on original data, augmented data, and
synthetic-only data generalize consistently to the same original holdout set.

A gradient boosting model and a compact PyTorch feedforward network are then trained on the winning
feature and data configuration and compared, with class weighting to account for the minority attack
class. SHAP explains the reference model, and the result is checked against Dorothy Denning's 1987
statistical model of intrusion detection: that intrusions manifest as statistical deviations from an
established behavioral baseline, not as fixed signatures. A small LangGraph agent turns a flagged
flow's prediction into a natural-language security recommendation, and an interactive function ties the
full pipeline together for a single candidate flow.

### **3.2 - The Rediscovery Standard**

The model is never given a single hand-built "is this an attack" feature, and it is never told
which of the many engineered features is the important one. It sees SYN counts, packet and byte rates,
destination-port fan-out, and per-source baselines as separate columns among many others, mixed with a
second, independent set of features an AI proposed without seeing any labels. The payoff of this
notebook is that SHAP applied to a model trained this way surfaces a small handful of statistical
deviation and behavioral features (rate deviation from baseline, destination-port fan-out, and
connection repetition) as dominant over raw payload and flag counts, which is the same finding Dorothy
Denning formalized in 1987: intrusions are detectable as abnormal deviations from a subject's
established behavior, not as instances of known attack signatures.

## **4 - A Brief History of Intrusion Detection**

### **4.1 - Overview**

Intrusion detection as a formal discipline began with a single insight, published before most of
the attacks it would later be used to catch had even been invented. Dorothy Denning's 1987 paper, "An
Intrusion-Detection Model" (IEEE Transactions on Software Engineering), proposed that a system could
detect intrusions by building a statistical profile of normal behavior for each subject (a user, a
host, a network connection) and flagging activity that deviated from that profile by enough to be
statistically abnormal. This model made no reference to specific attacks. It did not need to, since an
attack's defining property, in Denning's framing, is that it looks different from what came before, not
that it matches a catalog entry.

The industry initially moved the other direction. Signature-based systems, epitomized by Snort
(1998), matched packets against a growing database of known attack patterns. Signature matching is
precise and easy to explain, but it only detects what has already been seen and cataloged, and it is
blind to a slow variant of a known attack that no signature covers. Anomaly-based systems returned to
Denning's original framing through the 2000s, and the arrival of machine learning gave that framing a
practical implementation: instead of hand-coding a statistical model of "normal," a classifier learns
the boundary between normal and abnormal flows directly from labeled traffic. Modern flow-based
detection, the approach this notebook takes, sits squarely in that lineage. It uses telemetry
summarized at the flow level (duration, byte and packet counts, flag counts, timing), the same category
of features Denning's original model was built around, rather than deep packet inspection.

### **4.2 - Milestones in Intrusion Detection**

| Era | Milestone | Approach | Notes |
|---|---|---|---|
| 1980 | James Anderson's "Computer Security Threat Monitoring and Surveillance" | Audit-trail review | First formal proposal to use audit logs for threat detection |
| 1987 | Dorothy Denning, "An Intrusion-Detection Model" | Statistical anomaly detection | Established that intrusions are detectable as deviations from a subject's behavioral profile |
| 1990s | Early host-based and network-based IDS products (NIDES, NSM) | Statistical and rule-based hybrid | First operational anomaly-detection systems, largely research-lab deployments |
| 1998 | Snort released | Signature matching | Open-source, rule-based, becomes the de facto standard for signature IDS |
| 1999 | DARPA/KDD Cup 99 intrusion dataset | Early ML benchmark | First widely used labeled dataset for ML-based intrusion detection research |
| 2000s | Statistical and clustering-based anomaly detection matures | Unsupervised anomaly detection | Renewed interest in Denning's original framing as signature catalogs became unmanageable |
| 2010s | Flow-based network monitoring (NetFlow/IPFIX-derived features) becomes standard | Flow-level ML classifiers | Detection moves from payload inspection to summarized flow telemetry, the level this notebook operates at |
| 2020s | Behavioral baselining in commercial network detection and response (NDR) products | Per-entity statistical baselines plus ML | Modern commercial NDR products explicitly model a per-host or per-user baseline, a direct descendant of Denning's 1987 model |

The throughline from 1987 to the present is not a straight line, since the industry spent much of the
1990s and 2000s on signature matching before returning to statistical baselining at scale. The
throughline is the recurring conclusion that signatures alone do not generalize to new attacks, while a
well-built statistical deviation model does, because it does not need to have seen an attack before to
flag it.

## **5 - The Science: Statistical Deviation Detection and Network Traffic**

### **5.1 - Denning's Statistical Anomaly Model**

Denning's 1987 model represents each subject (in this notebook, each traffic source) by a set of
statistical profiles built from historical observations: measures such as the mean and variance of
login frequency, resource usage, or, in the network setting this notebook adapts the idea to, packet
rate, byte rate, and destination fan-out. A new observation is scored against the profile using a
distance measure such as:

`deviation = (observed - baseline_mean) / baseline_spread`

An observation whose deviation exceeds a threshold is flagged as anomalous. Denning's model treats the
threshold and the profile itself as adaptive, updated over time as behavior legitimately shifts, but the
core claim does not depend on that detail: intrusions are detectable because they are statistically
unusual relative to established behavior, not because they match a known pattern. This notebook does
not hand a deviation score to the model as an input. It gives the model the raw ingredients, current
packet rate, current byte rate, current destination fan-out, and each source's established baseline for
each, as separate columns, and checks in Section 15 whether the model's own explanation of itself
recovers Denning's deviation framing without being told to.

### **5.2 - Flow-Level Feature Definitions**

A network flow is the set of packets exchanged between a source and destination sharing a protocol
and port over a bounded time window. Flow-level monitoring (as opposed to full packet capture)
summarizes each flow into a small number of statistics, which is what makes it practical to run on
every connection crossing a network boundary. The features used in this notebook fall into four
groups: volume (duration, packet count, byte count), protocol behavior (SYN, ACK, FIN, and RST flag
counts, which reveal whether a TCP handshake completed normally), fan-out (how many distinct
destinations or ports one source contacts in a window, the signature of reconnaissance), and timing
(inter-arrival statistics, which distinguish human-paced traffic from machine-paced automation).

### **5.3 - Self-Similarity and Heavy-Tailed Traffic**

Leland, Taqqu, Willinger, and Wilson's 1994 measurement study of Ethernet traffic ("On the
Self-Similar Nature of Ethernet Traffic") established that real network traffic does not converge to
smooth, Poisson-like averages as it is aggregated over time. Instead, it is statistically self-similar
across time scales, a small number of very large flows ("elephant flows") carry a disproportionate
share of total bytes, while the large majority of flows ("mice flows") are small. This notebook's
synthetic benign traffic is generated from a heavy-tailed distribution for exactly this reason, so that
the exploratory data analysis in Section 8 shows the same qualitative shape (a small number of large
flows dominating total byte volume) that motivated decades of traffic-engineering and capacity-planning
work following Leland et al.'s finding. This finding is background for the exploratory analysis and
does not itself receive an explainability proof in this notebook.

### **5.4 - How the Three Attack Types Deviate from Baseline**

Each attack type in this dataset is a different statistical deviation from established behavior,
which is precisely why a single deviation-based model can catch all three without attack-specific
rules. A SYN flood deviates in rate: an overwhelming volume of connection-initiation packets with the
handshake never completing, so the SYN ratio approaches one and payload bytes approach zero. A port
scan deviates in fan-out: one source contacts an unusually large number of distinct destination ports
in a short window, while each individual probe carries almost no data. A brute-force login attempt
deviates in repetition and timing regularity: many short flows target the same destination port from
the same source, spaced at suspiciously regular intervals that a human typing a password would not
produce. Section 7 generates all three attack types directly from these statistical signatures.

## **6 - Installing and Importing the Libraries**

### **6.1 - Overview**

This notebook uses a standard scientific Python stack (numpy, pandas, scikit-learn, matplotlib,
seaborn, PyTorch), the SHAP explainability library, the Hugging Face free inference router through an
OpenAI-compatible client, and LangGraph for the agentic layer. The installation cell is safe to run
repeatedly, since pip skips packages that are already satisfied.

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# Install packages that are not preinstalled on a fresh Colab runtime or a bare local environment.
# This is safe to run repeatedly; pip skips packages that are already satisfied.
required = ["shap", "langgraph", "openai"]
for pkg in required:
    try:
        __import__(pkg if pkg != "openai" else "openai")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Environment ready. Running in Colab:", IN_COLAB)

In [ ]:
import os
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

import shap

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = "data"
PLOTS_DIR = "plots"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Hugging Face router client setup. This client is reused in Sections 9, 10, and 16.
# Read the token from the environment, or from Colab secrets if running on Colab.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

from openai import OpenAI

# The free Hugging Face Serverless Inference API is exposed through an OpenAI-compatible router.
# Model IDs occasionally need a provider suffix (e.g. "Qwen/Qwen2.5-1.5B-Instruct:together").
# Check https://huggingface.co/docs/api-inference/en/index for the current syntax if this call fails.
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

hf_client = None
if HF_TOKEN:
    hf_client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
    print("Hugging Face router client configured.")
else:
    print("No HF_TOKEN found. Set os.environ['HF_TOKEN'] to a free Hugging Face token to run "
          "the GenAI cells live. Fallback content will be used instead.")

def call_hf_llm(prompt, max_tokens=400, temperature=0.4):
    """Call the HF router chat completion endpoint with a graceful fallback on any failure."""
    if hf_client is None:
        return None
    try:
        response = hf_client.chat.completions.create(
            model=HF_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("HF router call failed, using fallback content instead. Error:", exc)
        return None

## **7 - Generating the Synthetic Network Flow Dataset**

### **7.1 - Overview**

No public dataset exposes raw flow telemetry at flow-level granularity alongside a clean,
non-proprietary ground-truth label for these four traffic classes, so this notebook generates a
synthetic dataset from statistical models of each class, calibrated qualitatively against the
signatures described in Section 5. For each of four classes (benign, SYN flood, port scan,
brute force), the generator samples flow duration, packet and byte counts, TCP flag counts,
destination-port fan-out, source-port entropy, connection repetition, and inter-arrival timing from
distributions chosen to reproduce that class's real-world statistical signature. Each row also carries
a per-source established baseline for packet rate, byte rate, and destination-port fan-out, sampled
independently of the row's own current behavior, exactly as Denning's model separates a subject's
historical profile from a new observation. The dataset totals 10,000 flow records: 8,000 benign and
roughly 2,000 attack flows split across the three attack types, an imbalance deliberately kept close
to what a real network boundary sees, where malicious flows are a small minority of total traffic.

### **7.2 - Benign Traffic Generation**

Benign flow duration and byte count are drawn from lognormal distributions, which produces the
heavy-tailed mix of many small "mice" flows and a few large "elephant" flows described in Section 5.3.
Packet counts scale with byte count plus independent noise. TCP flags reflect a normally completed
handshake (a small number of SYN and FIN packets, many ACKs). Destination ports are drawn from a
weighted list of common services (web, email, DNS, remote access), and each source's baseline is set
close to its own current behavior with small noise, since a benign source's current traffic is, by
construction, close to its own historical norm.

In [ ]:
COMMON_PORTS = [80, 443, 443, 443, 25, 587, 53, 22, 993, 21, 3389]

def sample_benign(n, rng):
    duration = rng.lognormal(mean=1.1, sigma=1.3, size=n)
    duration = np.clip(duration, 0.05, 600)

    byte_count = rng.lognormal(mean=8.5, sigma=1.8, size=n)
    byte_count = np.clip(byte_count, 80, 5_000_000)

    packet_count = np.clip((byte_count / rng.normal(650, 150, n)).round(), 2, 20000)

    syn_count = rng.integers(1, 3, n).astype(float)
    syn_count[rng.random(n) < 0.03] = 3
    fin_count = rng.integers(1, 3, n).astype(float)
    # A small share of benign flows end abruptly (client timeout, app-level reset) rather than
    # with a clean FIN, so fin_count is not a perfectly deterministic marker of "not an attack."
    fin_count[rng.random(n) < 0.05] = 0
    ack_count = np.clip(packet_count - syn_count - fin_count, 1, None)
    rst_count = (rng.random(n) < 0.03).astype(float)

    dst_port = rng.choice(COMMON_PORTS, size=n)
    unique_dst_ports_window = np.clip(rng.poisson(1.6, n) + 1, 1, 6)
    src_port_entropy_window = np.clip(rng.normal(3.6, 0.5, n), 1.0, 5.5)
    flows_per_src_dst_pair_window = np.clip(rng.poisson(1.2, n) + 1, 1, 6)

    inter_arrival_mean_ms = rng.lognormal(mean=4.2, sigma=0.9, size=n)
    inter_arrival_mean_ms = np.clip(inter_arrival_mean_ms, 20, 4000)
    inter_arrival_cv = np.clip(rng.normal(0.8, 0.25, n), 0.15, 2.0)

    packets_per_sec_actual = packet_count / duration
    bytes_per_sec_actual = byte_count / duration
    baseline_pkt_rate_src = np.clip(packets_per_sec_actual * rng.normal(1.0, 0.15, n), 0.05, None)
    baseline_byte_rate_src = np.clip(bytes_per_sec_actual * rng.normal(1.0, 0.15, n), 1.0, None)
    baseline_unique_ports_src = np.clip(unique_dst_ports_window * rng.normal(1.0, 0.2, n), 1.0, None)

    return pd.DataFrame({
        "attack_type": "benign",
        "flow_duration_s": duration,
        "packet_count": packet_count,
        "byte_count": byte_count,
        "syn_count": syn_count,
        "ack_count": ack_count,
        "fin_count": fin_count,
        "rst_count": rst_count,
        "dst_port": dst_port,
        "unique_dst_ports_window": unique_dst_ports_window,
        "src_port_entropy_window": src_port_entropy_window,
        "flows_per_src_dst_pair_window": flows_per_src_dst_pair_window,
        "inter_arrival_mean_ms": inter_arrival_mean_ms,
        "inter_arrival_cv": inter_arrival_cv,
        "baseline_pkt_rate_src": baseline_pkt_rate_src,
        "baseline_byte_rate_src": baseline_byte_rate_src,
        "baseline_unique_ports_src": baseline_unique_ports_src,
    })

### **7.3 - Attack Traffic Generation**

Each attack generator reproduces the statistical signature described in Section 5.4. The SYN-flood
generator produces an overwhelming SYN ratio and near-zero payload with almost no completed handshake.
The port-scan generator produces a very high destination-port fan-out per source in a short window,
with tiny per-probe packet and byte counts. The brute-force generator produces many short flows
repeated against the same single destination port, at unusually regular inter-arrival intervals. In
every case, the baseline columns are sampled from the same benign-like ranges used in Section 7.2,
representing each source's behavior before the attack began, so that the deviation between current
behavior and established baseline is large by construction, without that deviation ever being computed
or exposed as a single feature at generation time.

In [ ]:
def sample_syn_flood(n, rng):
    duration = np.clip(rng.lognormal(mean=-1.2, sigma=0.6, size=n), 0.01, 3.0)
    packet_count = np.clip(rng.lognormal(mean=6.5, sigma=0.7, size=n), 50, 50000)
    syn_count = np.clip(packet_count * rng.normal(0.97, 0.02, n), 1, None)
    ack_count = np.clip(packet_count - syn_count, 0, None) * rng.uniform(0, 0.3, n)
    fin_count = np.zeros(n)
    # A small share of flood flows complete a handshake before the flood saturates the target,
    # so a completed handshake alone is not a perfectly deterministic "not a flood" marker.
    fin_count[rng.random(n) < 0.02] = 1
    rst_count = np.clip(rng.poisson(0.5, n), 0, None).astype(float)
    byte_count = packet_count * rng.normal(58, 6, n)
    byte_count = np.clip(byte_count, 40 * packet_count, None)

    dst_port = rng.choice([80, 443, 22], size=n)
    unique_dst_ports_window = np.ones(n)
    src_port_entropy_window = np.clip(rng.normal(3.4, 0.6, n), 1.0, 5.5)
    flows_per_src_dst_pair_window = np.clip(rng.poisson(3, n) + 1, 1, 12)

    inter_arrival_mean_ms = np.clip(rng.lognormal(mean=0.3, sigma=0.6, size=n), 0.5, 50)
    inter_arrival_cv = np.clip(rng.normal(0.4, 0.15, n), 0.05, 1.2)

    baseline_pkt_rate_src = np.clip(rng.lognormal(mean=2.0, sigma=0.8, size=n), 0.1, 200)
    baseline_byte_rate_src = np.clip(rng.lognormal(mean=7.5, sigma=1.0, size=n), 10, 200000)
    baseline_unique_ports_src = np.clip(rng.poisson(1.6, n) + 1, 1, 6).astype(float)

    return pd.DataFrame({
        "attack_type": "syn_flood",
        "flow_duration_s": duration,
        "packet_count": packet_count,
        "byte_count": byte_count,
        "syn_count": syn_count,
        "ack_count": ack_count,
        "fin_count": fin_count,
        "rst_count": rst_count,
        "dst_port": dst_port,
        "unique_dst_ports_window": unique_dst_ports_window,
        "src_port_entropy_window": src_port_entropy_window,
        "flows_per_src_dst_pair_window": flows_per_src_dst_pair_window,
        "inter_arrival_mean_ms": inter_arrival_mean_ms,
        "inter_arrival_cv": inter_arrival_cv,
        "baseline_pkt_rate_src": baseline_pkt_rate_src,
        "baseline_byte_rate_src": baseline_byte_rate_src,
        "baseline_unique_ports_src": baseline_unique_ports_src,
    })

def sample_port_scan(n, rng):
    duration = np.clip(rng.lognormal(mean=-0.5, sigma=0.7, size=n), 0.02, 20)
    packet_count = np.clip(rng.normal(2.2, 0.8, n).round(), 1, 5)
    byte_count = packet_count * rng.normal(55, 10, n)
    byte_count = np.clip(byte_count, 40, 400)

    syn_count = rng.choice([1.0, 2.0], size=n, p=[0.85, 0.15])
    ack_count = np.zeros(n)
    fin_count = np.zeros(n)
    # A handful of probes hit an open port and complete a full connect scan rather than a bare
    # SYN probe, so fin_count is not a perfectly deterministic marker of scan traffic.
    fin_count[rng.random(n) < 0.04] = 1
    rst_count = np.clip(rng.integers(0, 2, n), 0, 1).astype(float)

    dst_port = rng.integers(1, 65535, n)
    unique_dst_ports_window = np.clip(rng.lognormal(mean=4.3, sigma=0.6, size=n), 25, 2000)
    src_port_entropy_window = np.clip(rng.normal(1.8, 0.5, n), 0.2, 4.0)
    flows_per_src_dst_pair_window = np.ones(n)

    inter_arrival_mean_ms = np.clip(rng.lognormal(mean=2.5, sigma=0.5, size=n), 2, 300)
    inter_arrival_cv = np.clip(rng.normal(0.3, 0.1, n), 0.05, 0.8)

    baseline_pkt_rate_src = np.clip(rng.lognormal(mean=1.2, sigma=0.7, size=n), 0.05, 50)
    baseline_byte_rate_src = np.clip(rng.lognormal(mean=6.0, sigma=0.9, size=n), 5, 20000)
    baseline_unique_ports_src = np.clip(rng.poisson(1.6, n) + 1, 1, 6).astype(float)

    return pd.DataFrame({
        "attack_type": "port_scan",
        "flow_duration_s": duration,
        "packet_count": packet_count,
        "byte_count": byte_count,
        "syn_count": syn_count,
        "ack_count": ack_count,
        "fin_count": fin_count,
        "rst_count": rst_count,
        "dst_port": dst_port,
        "unique_dst_ports_window": unique_dst_ports_window,
        "src_port_entropy_window": src_port_entropy_window,
        "flows_per_src_dst_pair_window": flows_per_src_dst_pair_window,
        "inter_arrival_mean_ms": inter_arrival_mean_ms,
        "inter_arrival_cv": inter_arrival_cv,
        "baseline_pkt_rate_src": baseline_pkt_rate_src,
        "baseline_byte_rate_src": baseline_byte_rate_src,
        "baseline_unique_ports_src": baseline_unique_ports_src,
    })

def sample_brute_force(n, rng):
    duration = np.clip(rng.lognormal(mean=-0.3, sigma=0.5, size=n), 0.05, 8)
    packet_count = np.clip(rng.normal(14, 4, n).round(), 6, 40)
    byte_count = np.clip(rng.normal(900, 300, n), 150, 4000)

    syn_count = np.ones(n)
    syn_count[rng.random(n) < 0.08] = 2
    fin_count = np.ones(n)
    # A share of login attempts fail and reset instead of closing cleanly, so a completed
    # handshake alone does not perfectly separate brute-force flows from benign ones.
    fin_count[rng.random(n) < 0.06] = 0
    ack_count = np.clip(packet_count - syn_count - fin_count, 1, None)
    rst_count = (rng.random(n) < 0.05).astype(float)

    dst_port = rng.choice([22, 3389], size=n)
    unique_dst_ports_window = np.ones(n)
    src_port_entropy_window = np.clip(rng.normal(3.3, 0.5, n), 1.0, 5.0)
    flows_per_src_dst_pair_window = np.clip(rng.lognormal(mean=3.2, sigma=0.5, size=n), 15, 400)

    inter_arrival_mean_ms = np.clip(rng.normal(450, 100, n), 100, 900)
    inter_arrival_cv = np.clip(rng.normal(0.15, 0.06, n), 0.02, 0.4)

    baseline_pkt_rate_src = np.clip(rng.lognormal(mean=1.0, sigma=0.6, size=n), 0.05, 30)
    baseline_byte_rate_src = np.clip(rng.lognormal(mean=5.5, sigma=0.8, size=n), 5, 8000)
    baseline_unique_ports_src = np.clip(rng.poisson(1.3, n) + 1, 1, 5).astype(float)

    return pd.DataFrame({
        "attack_type": "brute_force",
        "flow_duration_s": duration,
        "packet_count": packet_count,
        "byte_count": byte_count,
        "syn_count": syn_count,
        "ack_count": ack_count,
        "fin_count": fin_count,
        "rst_count": rst_count,
        "dst_port": dst_port,
        "unique_dst_ports_window": unique_dst_ports_window,
        "src_port_entropy_window": src_port_entropy_window,
        "flows_per_src_dst_pair_window": flows_per_src_dst_pair_window,
        "inter_arrival_mean_ms": inter_arrival_mean_ms,
        "inter_arrival_cv": inter_arrival_cv,
        "baseline_pkt_rate_src": baseline_pkt_rate_src,
        "baseline_byte_rate_src": baseline_byte_rate_src,
        "baseline_unique_ports_src": baseline_unique_ports_src,
    })

rng = np.random.default_rng(42)
N_BENIGN, N_SYN, N_SCAN, N_BRUTE = 8000, 700, 700, 600

frames = [
    sample_benign(N_BENIGN, rng),
    sample_syn_flood(N_SYN, rng),
    sample_port_scan(N_SCAN, rng),
    sample_brute_force(N_BRUTE, rng),
]
flows = pd.concat(frames, ignore_index=True)
flows["label"] = (flows["attack_type"] != "benign").astype(int)
flows = flows.sample(frac=1, random_state=42).reset_index(drop=True)
flows["flow_id"] = [f"FLOW-{i:06d}" for i in range(len(flows))]

flows = flows[[
    "flow_id", "attack_type", "label", "flow_duration_s", "packet_count", "byte_count",
    "syn_count", "ack_count", "fin_count", "rst_count", "dst_port", "unique_dst_ports_window",
    "src_port_entropy_window", "flows_per_src_dst_pair_window", "inter_arrival_mean_ms",
    "inter_arrival_cv", "baseline_pkt_rate_src", "baseline_byte_rate_src", "baseline_unique_ports_src",
]]

flows.to_csv(os.path.join(DATA_DIR, "synthetic_network_flows.csv"), index=False)
print("Generated", len(flows), "synthetic flow records.")
flows.head()

### **7.4 - Class Balance Check**

Before proceeding, it is worth confirming the class balance matches the roughly 80/20 benign to
malicious split the generator targets, and that each attack type is represented with enough rows to
train and evaluate on.

In [ ]:
class_summary = flows["attack_type"].value_counts()
print(class_summary)
print("\nOverall malicious share:", round(flows["label"].mean(), 3))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=class_summary.index, y=class_summary.values, ax=ax, color="steelblue")
ax.set_title("Flow Count by Traffic Class")
ax.set_ylabel("Flow count")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "class_balance.png"), dpi=100)
plt.show()

## **8 - Exploratory Data Analysis and Human-Engineered Features**

### **8.1 - Overview**

This section first looks at the raw distribution of benign flow size and duration, to check the
heavy-tailed shape described in Section 5.3, and then builds the six features a network security
analyst would compute by hand before ever training a model.

In [ ]:
benign = flows[flows["attack_type"] == "benign"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(benign["byte_count"], bins=60, color="steelblue")
axes[0].set_title("Benign Flow Byte Count (linear scale)")
axes[0].set_xlabel("Bytes")

log_bins = np.logspace(np.log10(benign["byte_count"].min()), np.log10(benign["byte_count"].max()), 50)
axes[1].hist(benign["byte_count"], bins=log_bins, color="indianred")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Benign Flow Byte Count (log-log scale)")
axes[1].set_xlabel("Bytes (log scale)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "heavy_tailed_flows.png"), dpi=100)
plt.show()

top_decile_share = (
    benign.sort_values("byte_count", ascending=False).head(int(len(benign) * 0.1))["byte_count"].sum()
    / benign["byte_count"].sum()
)
print(f"Top 10% of benign flows by size carry {top_decile_share:.1%} of total benign bytes, "
      "the elephant-flow pattern reported by Leland et al. (1994).")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
num_cols = ["flow_duration_s", "packet_count", "byte_count", "syn_count",
            "unique_dst_ports_window", "inter_arrival_cv"]
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=flows, x="attack_type", y=col, ax=ax)
    ax.set_yscale("log")
    ax.set_title(col)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "eda_by_class.png"), dpi=100)
plt.show()

### **8.2 - Human-Engineered Feature 1 and 2: SYN Ratio and RST Ratio**

SYN ratio is the share of a flow's packets that are SYN flags, the classic flood indicator since a
normal handshake produces one SYN per connection while a flood produces almost nothing else. RST ratio
captures how often a flow ends in a reset rather than a normal close, a secondary signature of scan
traffic hitting closed ports.

### **8.3 - Human-Engineered Feature 3 and 4: Packet Rate and Byte Rate**

Packets per second and bytes per second are the most basic throughput measures a monitoring
appliance computes, and both are pushed to extremes by flooding traffic relative to normal use.

### **8.4 - Human-Engineered Feature 5 and 6: Byte-to-Packet Ratio and Port Fan-Out Rate**

Byte-to-packet ratio separates flows carrying real payload from flows carrying almost none, the
signature shared by SYN floods and port-scan probes. Port fan-out rate normalizes destination-port
fan-out by flow duration, distinguishing a fast automated scan from a slower, more human-paced pattern
of contacting a handful of services.

### **8.5 - Human-Engineered Feature 7 and 8: Connection Repetition and Timing Regularity**

Connection repetition is the ratio of repeat flows to the same destination and port, relative to
the source's overall port fan-out, the signature of a brute-force campaign hammering a single service.
Timing regularity is the inverse of the inter-arrival coefficient of variation, since automated
attack tools tend to pace requests far more regularly than a human or a bursty legitimate application
would.

### **8.6 - Human-Engineered Feature 9, 10, and 11: Deviation from Source Baseline**

These three features compute how far a flow's current packet rate, byte rate, and destination-port
fan-out sit from that source's own established baseline, the same historical-profile-versus-observation
comparison at the center of Denning's 1987 model. None of these features is labeled as the answer to
the classification problem; they are built alongside eight other candidate features and left for the
model, and later its explanation, to weigh on its own merits in Section 14 and Section 15.

In [ ]:
def add_human_engineered_features(df):
    df = df.copy()
    eps = 1e-6

    df["syn_ratio"] = df["syn_count"] / (df["packet_count"] + eps)
    df["rst_ratio"] = df["rst_count"] / (df["packet_count"] + eps)

    df["packets_per_sec"] = df["packet_count"] / (df["flow_duration_s"] + eps)
    df["bytes_per_sec"] = df["byte_count"] / (df["flow_duration_s"] + eps)

    df["bytes_per_packet"] = df["byte_count"] / (df["packet_count"] + eps)
    df["port_fanout_rate"] = df["unique_dst_ports_window"] / (df["flow_duration_s"] + eps)

    df["connection_repeat_ratio"] = (
        df["flows_per_src_dst_pair_window"] / (df["unique_dst_ports_window"] + 1)
    )
    df["timing_regularity"] = 1.0 / (df["inter_arrival_cv"] + eps)

    df["pkt_rate_deviation"] = (
        (df["packets_per_sec"] - df["baseline_pkt_rate_src"]) / (df["baseline_pkt_rate_src"] + eps)
    )
    df["byte_rate_deviation"] = (
        (df["bytes_per_sec"] - df["baseline_byte_rate_src"]) / (df["baseline_byte_rate_src"] + eps)
    )
    df["port_count_deviation"] = (
        (df["unique_dst_ports_window"] - df["baseline_unique_ports_src"])
        / (df["baseline_unique_ports_src"] + eps)
    )
    return df

HUMAN_FEATURES = [
    "syn_ratio", "rst_ratio", "packets_per_sec", "bytes_per_sec", "bytes_per_packet",
    "port_fanout_rate", "connection_repeat_ratio", "timing_regularity",
    "pkt_rate_deviation", "byte_rate_deviation", "port_count_deviation",
]

flows_feat = add_human_engineered_features(flows)
flows_feat[["attack_type"] + HUMAN_FEATURES].groupby("attack_type").median().round(3)

## **9 - GenAI-Generated Features**

### **9.1 - Overview**

The eleven human-engineered features in Section 8 encode standard network security analyst
practice: ratios, rates, and deviation-from-baseline measures. A free, small, open-weight language
model is called through the Hugging Face router and asked to propose an independent set of candidate
features from the raw schema alone, without seeing the human-engineered set or any labels. If the
HF router call fails or no token is configured, a fixed fallback list, written in the same style an
LLM commonly proposes, is used instead, so the rest of the notebook runs unchanged either way.

In [ ]:
FEATURE_PROMPT = """You are assisting with feature engineering for a network intrusion detection
dataset. The target variable is a binary label, malicious or benign, named label.

The raw columns available are:
- flow_duration_s: duration of the flow in seconds
- packet_count: total packets in the flow
- byte_count: total bytes in the flow
- syn_count, ack_count, fin_count, rst_count: TCP flag counts in the flow
- dst_port: destination port number
- unique_dst_ports_window: distinct destination ports contacted by this source in the window
- src_port_entropy_window: entropy of source ports used by this source in the window
- flows_per_src_dst_pair_window: flows from this source to this destination and port in the window
- inter_arrival_mean_ms, inter_arrival_cv: mean and coefficient of variation of packet inter-arrival time
- baseline_pkt_rate_src, baseline_byte_rate_src, baseline_unique_ports_src: this source's established
  historical baseline for packet rate, byte rate, and destination-port fan-out

Propose 6 additional candidate engineered features as nonlinear transforms or interaction terms of
these raw columns, that might help a model distinguish malicious from benign flows. Do not propose a
feature that is simply (packet_count or byte_count) divided by baseline_pkt_rate_src or
baseline_byte_rate_src directly.

Respond with ONLY a JSON array, no other text, where each element has this exact shape:
{"name": "short_snake_case_name", "expression": "python expression using the raw column names above"}
"""

llm_response = call_hf_llm(FEATURE_PROMPT, max_tokens=500, temperature=0.3)

# Fallback candidate features, used if the HF call is unavailable. These are written in the same
# style an LLM commonly proposes: log/sqrt transforms and cross-parameter interaction terms.
FALLBACK_AI_FEATURES = [
    {"name": "log_packet_count", "expression": "np.log1p(packet_count)"},
    {"name": "log_byte_count", "expression": "np.log1p(byte_count)"},
    {"name": "log_duration", "expression": "np.log1p(flow_duration_s)"},
    {"name": "duration_port_interaction", "expression": "flow_duration_s * unique_dst_ports_window"},
    {"name": "sqrt_flows_per_pair", "expression": "np.sqrt(flows_per_src_dst_pair_window)"},
    {"name": "entropy_over_duration", "expression": "src_port_entropy_window / (flow_duration_s + 1e-6)"},
]

def parse_ai_features(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "name" in item and "expression" in item
        return parsed
    except Exception:
        return None

ai_feature_specs = parse_ai_features(llm_response)
if ai_feature_specs is None:
    print("Using fallback AI-suggested feature list (no live HF response parsed).")
    ai_feature_specs = FALLBACK_AI_FEATURES
else:
    print("Parsed", len(ai_feature_specs), "AI-suggested features from the HF router response.")

for spec in ai_feature_specs:
    print(" -", spec["name"], ":", spec["expression"])

### **9.2 - Constructing the AI-Suggested Features**

Each proposed expression is evaluated against the raw dataframe columns inside a restricted
namespace (only numpy and the raw numeric columns are exposed), so a malformed or unexpected expression
fails safely for that one feature rather than crashing the notebook.

In [ ]:
def build_ai_features(df, specs):
    df = df.copy()
    safe_globals = {"np": np}
    built_names = []
    for spec in specs:
        try:
            local_vars = {col: df[col].values for col in df.columns if df[col].dtype != object}
            value = eval(spec["expression"], safe_globals, local_vars)
            df[spec["name"]] = value
            built_names.append(spec["name"])
        except Exception as exc:
            print("Skipping feature", spec["name"], "due to error:", exc)
    return df, built_names

flows_feat, AI_FEATURES = build_ai_features(flows_feat, ai_feature_specs)
flows_feat[AI_FEATURES].describe().round(3)

### **9.3 - Ablation: Human Features vs. AI Features vs. Combined**

The ablation trains the same gradient boosting classifier on three feature sets, all of which
include the raw telemetry columns as a common baseline: raw features plus the human-engineered set, raw
features plus the AI-suggested set, and raw features plus both. Comparing against a common raw-feature
baseline isolates the incremental value of each engineered feature source. F1 and ROC-AUC are used
rather than accuracy, since malicious flows are a minority class.

In [ ]:
RAW_FEATURES = [
    "flow_duration_s", "packet_count", "byte_count", "syn_count", "ack_count", "fin_count",
    "rst_count", "dst_port", "unique_dst_ports_window", "src_port_entropy_window",
    "flows_per_src_dst_pair_window", "inter_arrival_mean_ms", "inter_arrival_cv",
    "baseline_pkt_rate_src", "baseline_byte_rate_src", "baseline_unique_ports_src",
]

TARGET = "label"

train_idx, test_idx = train_test_split(
    flows_feat.index, test_size=0.2, random_state=42, stratify=flows_feat[TARGET]
)

def evaluate_feature_set(feature_cols, label):
    X_train = flows_feat.loc[train_idx, feature_cols]
    X_test = flows_feat.loc[test_idx, feature_cols]
    y_train = flows_feat.loc[train_idx, TARGET]
    y_test = flows_feat.loc[test_idx, TARGET]

    model = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.08,
                                        random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    return {"feature_set": label, "n_features": len(feature_cols), "f1": f1, "roc_auc": auc}

ablation_results = [
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES, "Raw + Human-engineered"),
    evaluate_feature_set(RAW_FEATURES + AI_FEATURES, "Raw + AI-suggested"),
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES, "Raw + Human + AI (combined)"),
]
ablation_df = pd.DataFrame(ablation_results)
ablation_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=ablation_df, x="feature_set", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("F1 Score by Feature Set")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=ablation_df, x="feature_set", y="roc_auc", ax=axes[1], color="indianred")
axes[1].set_title("ROC-AUC by Feature Set")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "feature_ablation.png"), dpi=100)
plt.show()

print(
    "The combined feature set is expected to match or improve on both the human-only and AI-only "
    "sets, since it has strictly more information available, and the two sources encode different "
    "aspects of the traffic: the human set targets known deviation-from-baseline and flag-ratio "
    "signatures, and the AI set targets nonlinear transforms and interaction terms not derived from "
    "intrusion-detection theory."
)

## **10 - GenAI Synthetic Data Augmentation**

### **10.1 - Overview**

A language model is not a reliable source of numerically precise, physically consistent rows of
network telemetry. Asking it to hallucinate exact packet and byte counts directly would silently
corrupt the ground truth. A more defensible use of the model is to ask it to identify which regions of
the traffic space are thin in the existing data and propose realistic sampling parameters for those
regions, in structured JSON. The actual synthetic rows are then sampled from those LLM-proposed
parameters using the same generation functions from Section 7, so the ground truth stays intact and
only the sampling strategy comes from the language model.

The two regions flagged here are slow, low-rate port scans (an attacker deliberately probing one port
every few seconds to stay under rate-based thresholds, which the original generator undersamples since
it draws inter-arrival time from a single range) and unusually large benign transfers (bulk file
transfer or backup traffic, an edge of the benign envelope that a naive monitoring rule might mistake
for exfiltration).

In [ ]:
AUGMENT_PROMPT = """You are helping design a synthetic data augmentation plan for a network
intrusion detection dataset. The existing dataset undersamples two regions of the traffic space:

1. Slow, low-and-slow port scans: an attacker probing one destination port every several seconds
   instead of rapid-fire scanning, to stay under simple rate thresholds.
2. Very large benign bulk transfers: legitimate backup or large file-transfer traffic at the extreme
   upper end of normal byte counts and duration.

For each region, propose a realistic sampling plan appropriate for that traffic type. Respond with
ONLY a JSON array of exactly 2 objects, no other text, in this exact shape:
{"region": "short label", "kind": "port_scan or benign", "n_rows": 200,
 "inter_arrival_mean_ms_mean": <float>, "inter_arrival_mean_ms_std": <float>,
 "unique_dst_ports_mean": <float>, "unique_dst_ports_std": <float>,
 "byte_count_log_mean": <float>, "byte_count_log_std": <float>}
"""

augment_response = call_hf_llm(AUGMENT_PROMPT, max_tokens=500, temperature=0.3)

# Fallback augmentation plan, used if the HF call is unavailable.
FALLBACK_AUGMENT_PLAN = [
    {"region": "Low-and-slow port scan", "kind": "port_scan", "n_rows": 200,
     "inter_arrival_mean_ms_mean": 4500, "inter_arrival_mean_ms_std": 900,
     "unique_dst_ports_mean": 40, "unique_dst_ports_std": 12,
     "byte_count_log_mean": 4.2, "byte_count_log_std": 0.4},
    {"region": "Large benign bulk transfer", "kind": "benign", "n_rows": 200,
     "inter_arrival_mean_ms_mean": 60, "inter_arrival_mean_ms_std": 20,
     "unique_dst_ports_mean": 1.5, "unique_dst_ports_std": 0.5,
     "byte_count_log_mean": 14.5, "byte_count_log_std": 1.0},
]

def parse_augment_plan(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "kind" in item and item["kind"] in ("port_scan", "benign")
        return parsed
    except Exception:
        return None

augment_plan = parse_augment_plan(augment_response)
if augment_plan is None:
    print("Using fallback augmentation plan (no live HF response parsed).")
    augment_plan = FALLBACK_AUGMENT_PLAN
else:
    print("Parsed", len(augment_plan), "augmentation regions from the HF router response.")

for region in augment_plan:
    print(" -", region["region"], ":", region["n_rows"], "rows of", region["kind"])

### **10.2 - Sampling Synthetic Rows from the LLM-Proposed Parameters**

Each region is sampled with the same generation functions used in Section 7, overriding only the
parameters the language model was asked to propose, so the resulting rows remain internally consistent
flow records rather than arbitrary numbers.

In [ ]:
def sample_augmented_rows(region, rng):
    n = region["n_rows"]
    if region["kind"] == "port_scan":
        base = sample_port_scan(n, rng)
        base["inter_arrival_mean_ms"] = np.clip(
            rng.normal(region["inter_arrival_mean_ms_mean"], region["inter_arrival_mean_ms_std"], n),
            50, 20000,
        )
        base["unique_dst_ports_window"] = np.clip(
            rng.normal(region["unique_dst_ports_mean"], region["unique_dst_ports_std"], n), 15, 3000,
        )
    else:
        base = sample_benign(n, rng)
        base["byte_count"] = np.clip(
            rng.lognormal(region["byte_count_log_mean"], region["byte_count_log_std"], n),
            50_000, 20_000_000,
        )
        base["packet_count"] = np.clip((base["byte_count"] / rng.normal(650, 150, n)).round(), 10, None)
        base["ack_count"] = np.clip(base["packet_count"] - base["syn_count"] - base["fin_count"], 1, None)
    return base

aug_rng = np.random.default_rng(123)
augmented_frames = [sample_augmented_rows(region, aug_rng) for region in augment_plan]
augmented_flows = pd.concat(augmented_frames, ignore_index=True)
augmented_flows["label"] = (augmented_flows["attack_type"] != "benign").astype(int)
augmented_flows = add_human_engineered_features(augmented_flows)
augmented_flows, _ = build_ai_features(augmented_flows, ai_feature_specs)
print("Generated", len(augmented_flows), "augmented rows across", len(augment_plan), "regions.")

### **10.3 - Three-Way Generalization Check**

The three-way check compares three training regimes, all evaluated against the same fixed original
holdout set. "Original only" trains on the original training split alone. "Original plus augmented"
mixes in the 400 targeted rows from the two thin regions above. "Full synthetic resample" trains on an
entirely fresh draw from the same four generation functions (a different random seed, no row shared
with the original training split, standing in for a team that trains only on simulator output rather
than any captured original flow), mixed with the same 400 targeted rows. A tolerance of 0.03 on F1 is
used: if the augmented and fully synthetic conditions stay within that tolerance of the original-only
model, the synthetic data is judged to preserve generalization rather than distort it.

In [ ]:
FULL_FEATURES = RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES

original_train = flows_feat.loc[train_idx]
original_test = flows_feat.loc[test_idx]

X_test_orig = original_test[FULL_FEATURES]
y_test_orig = original_test[TARGET]

def train_and_score(train_df, label):
    X_train = train_df[FULL_FEATURES]
    y_train = train_df[TARGET]
    model = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.08,
                                        random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test_orig)
    probs = model.predict_proba(X_test_orig)[:, 1]
    return {
        "condition": label,
        "f1": f1_score(y_test_orig, preds),
        "roc_auc": roc_auc_score(y_test_orig, probs),
    }, model

# A fully independent synthetic resample, drawn from the same four generation functions with a new
# seed, covering all four traffic classes rather than only the two thin regions from Section 10.2.
resample_rng = np.random.default_rng(777)
full_resample = pd.concat([
    sample_benign(N_BENIGN, resample_rng),
    sample_syn_flood(N_SYN, resample_rng),
    sample_port_scan(N_SCAN, resample_rng),
    sample_brute_force(N_BRUTE, resample_rng),
], ignore_index=True)
full_resample["label"] = (full_resample["attack_type"] != "benign").astype(int)
full_resample = add_human_engineered_features(full_resample)
full_resample, _ = build_ai_features(full_resample, ai_feature_specs)

mixed_train = pd.concat([original_train, augmented_flows], ignore_index=True)
synthetic_train = pd.concat([full_resample, augmented_flows], ignore_index=True)

result_original, _ = train_and_score(original_train, "Original only")
result_mixed, _ = train_and_score(mixed_train, "Original + Augmented")
result_synthetic, _ = train_and_score(synthetic_train, "Full synthetic resample")

generalization_df = pd.DataFrame([result_original, result_mixed, result_synthetic])
generalization_df["f1_gap_vs_original"] = (
    generalization_df["f1"] - generalization_df.loc[0, "f1"]
).abs()
generalization_df

In [ ]:
TOLERANCE = 0.03
within_tolerance = (generalization_df["f1_gap_vs_original"] <= TOLERANCE).all()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=generalization_df, x="condition", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("F1 on Fixed Original Holdout")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=generalization_df, x="condition", y="roc_auc", ax=axes[1], color="indianred")
axes[1].set_title("ROC-AUC on Fixed Original Holdout")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "generalization_check.png"), dpi=100)
plt.show()

print(f"All conditions within {TOLERANCE} F1 of original-only:", within_tolerance)
print(
    "Staying within tolerance across all three conditions indicates the LLM-proposed sampling "
    "parameters describe genuinely realistic regions of the traffic space, rather than a distribution "
    "the original physics-based generator would never have produced."
)

## **11 - Classical ML Model: Gradient Boosting and Random Forest**

### **11.1 - Overview**

This section trains the classical models that carry forward through the rest of the notebook,
using the winning feature configuration from Section 9 (raw plus human plus AI features) and the
original-only training data from Section 10, since the generalization check confirmed the augmented
data does not materially change performance and the original data is the more defensible training set
for the final model. Both a gradient boosting classifier and a random forest classifier use balanced
class weighting to account for malicious flows being a minority class, rather than a resampling scheme
such as SMOTE, which is unnecessary at this class ratio and adds a layer of synthetic interpolation on
top of an already-synthetic dataset.

In [ ]:
X_train_final = original_train[FULL_FEATURES]
y_train_final = original_train[TARGET]
X_test_final = original_test[FULL_FEATURES]
y_test_final = original_test[TARGET]

gbr_model = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                        random_state=42)
gbr_model.fit(X_train_final, y_train_final)

rf_model = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight="balanced",
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train_final, y_train_final)

def score_model(model, name):
    preds = model.predict(X_test_final)
    probs = model.predict_proba(X_test_final)[:, 1]
    return {
        "model": name,
        "f1": f1_score(y_test_final, preds),
        "precision": precision_score(y_test_final, preds),
        "recall": recall_score(y_test_final, preds),
        "roc_auc": roc_auc_score(y_test_final, probs),
    }

classical_results = pd.DataFrame([
    score_model(gbr_model, "Gradient Boosting"),
    score_model(rf_model, "Random Forest"),
])
classical_results

### **11.2 - Selecting the Reference Classical Model**

Gradient boosting and random forest typically land within a small margin of each other on tabular
data of this kind. The model with the higher F1 score is carried forward as the reference classical
model for the deep learning comparison, SHAP explanation, and agentic layer sections that follow.

In [ ]:
classical_model = gbr_model if classical_results.loc[0, "f1"] >= classical_results.loc[1, "f1"] else rf_model
classical_model_name = "Gradient Boosting" if classical_model is gbr_model else "Random Forest"
print("Reference classical model:", classical_model_name)

cm = confusion_matrix(y_test_final, classical_model.predict(X_test_final))
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Benign", "Malicious"], yticklabels=["Benign", "Malicious"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"{classical_model_name}: Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "classical_confusion_matrix.png"), dpi=100)
plt.show()

## **12 - Light Deep Learning Model: PyTorch Feedforward Network**

### **12.1 - Overview**

This section trains a compact feedforward neural network on the same feature set and data split as
the classical models, using PyTorch. The network is deliberately small: flow-level intrusion
classification from a couple dozen engineered features does not require a deep or wide network, and an
oversized network would only add overfitting risk without improving on a well-tuned tree ensemble. The
training loop is written with `device = "cuda" if torch.cuda.is_available() else "cpu"`, so the same
code runs on a Colab T4 GPU or a local CPU without modification. Class imbalance is handled with a
positive class weight inside the loss function, matching the balanced weighting used by the classical
models.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final.values)
X_test_scaled = scaler.transform(X_test_final.values)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train_final.values, dtype=torch.float32).view(-1, 1).to(DEVICE)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)

class IntrusionNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

intrusion_net = IntrusionNet(X_train_t.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(intrusion_net.parameters(), lr=1e-3, weight_decay=1e-5)

pos_weight_value = (y_train_final == 0).sum() / max((y_train_final == 1).sum(), 1)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE))

EPOCHS = 200
BATCH_SIZE = 128
n_samples = X_train_t.shape[0]
history = []

for epoch in range(EPOCHS):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    for start in range(0, n_samples, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        logits = intrusion_net(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.shape[0]

    history.append(epoch_loss / n_samples)
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}, train BCE loss: {history[-1]:.5f}")

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Training BCE loss")
plt.title("IntrusionNet Training Curve")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "nn_training_curve.png"), dpi=100)
plt.show()

### **12.2 - Comparing the Neural Network to the Classical Model**

Predictions from the network are converted from logits to probabilities with a sigmoid before
thresholding at 0.5, so the comparison to the classical model in Section 11 is on the same footing.

In [ ]:
intrusion_net.eval()
with torch.no_grad():
    nn_logits = intrusion_net(X_test_t).cpu().numpy().ravel()
nn_probs = 1 / (1 + np.exp(-nn_logits))
nn_preds = (nn_probs >= 0.5).astype(int)

nn_f1 = f1_score(y_test_final, nn_preds)
nn_auc = roc_auc_score(y_test_final, nn_probs)

model_comparison = pd.concat([
    classical_results[["model", "f1", "roc_auc"]],
    pd.DataFrame([{"model": "PyTorch FeedForward NN", "f1": nn_f1, "roc_auc": nn_auc}]),
], ignore_index=True)
model_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=model_comparison, x="model", y="f1", ax=ax, color="slateblue")
ax.set_title("F1 Score by Model")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "model_comparison.png"), dpi=100)
plt.show()

print(
    "The tree ensemble and the compact feedforward network land within a similar accuracy band on "
    "this dataset. The tree ensemble is retained as the reference model for SHAP in Section 14, since "
    "SHAP's TreeExplainer computes exact attributions for tree ensembles efficiently, and the network "
    "serves as a cross-check that the law rediscovered in Section 15 is not an artifact of one "
    "particular model family."
)

## **13 - Foundation Model Benchmark: TabPFN**

### **13.1 - Overview**

A pretrained tabular foundation model is trained once, on millions of synthetic tabular
learning tasks, and then applied to a new dataset without a per-dataset training step of its own; it
conditions on the labeled training rows it is given at inference time and predicts label probabilities
for new rows directly, rather than fitting parameters to this dataset the way the classical models in
Section 11 and the feedforward network in Section 12 do. TabPFN is a foundation model of this kind,
distributed as a free, installable Python package. This section fits TabPFN on the same feature set
used by the classical models and the feedforward network and evaluates it on the same held-out test
set, adding it as a third point of comparison alongside classical ML and light deep learning. TabPFN is
documented as comfortable up to roughly 10,000 rows and 500 features. This notebook's training split
holds about 8,000 rows, near that practical edge, so the training set is stratified down to a smaller
size before fitting TabPFN, a decision addressed directly in Section 13.2 rather than applied silently.

In [ ]:
# TabPFN is optional. Install and import it defensively, and skip the rest of this section with a
# clear message if it cannot be installed, rather than halting the notebook.
TABPFN_AVAILABLE = False
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
    print("TabPFN import succeeded.")
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)
        from tabpfn import TabPFNClassifier
        TABPFN_AVAILABLE = True
        print("TabPFN installed and imported successfully.")
    except Exception as exc:
        print("TabPFN could not be installed or imported; skipping the foundation model benchmark. "
              "Error:", exc)

### **13.2 - Fitting TabPFN on a Subsampled Training Set**

The training set is stratified down from about 8,000 rows to 3,000 rows, preserving the roughly
80/20 benign-to-malicious split, before fitting `TabPFNClassifier`. The full original test set, about
2,000 rows, is still used for evaluation, so the comparison in Section 13.3 runs against the same
held-out flows used throughout the rest of this notebook. Subsampling trades away some training rows
for a safety margin inside TabPFN's documented operating range and for a shorter fit time on a free-tier
or CPU-only runtime. If TabPFN's score lands close to the classical model's despite training on well
under half the rows the classical model saw, that gap is itself informative about how much a pretrained
prior can substitute for per-dataset training data.

In [ ]:
TABPFN_TRAIN_SIZE = 3000
tabpfn_results = None

if TABPFN_AVAILABLE:
    try:
        if len(X_train_final) > TABPFN_TRAIN_SIZE:
            X_train_tabpfn, _, y_train_tabpfn, _ = train_test_split(
                X_train_final, y_train_final, train_size=TABPFN_TRAIN_SIZE,
                stratify=y_train_final, random_state=42,
            )
        else:
            X_train_tabpfn, y_train_tabpfn = X_train_final, y_train_final

        print(f"Fitting TabPFN on {len(X_train_tabpfn)} stratified-subsampled rows "
              f"(malicious share {y_train_tabpfn.mean():.3f}), evaluating on the full "
              f"{len(X_test_final)}-row test set.")

        tabpfn_model = TabPFNClassifier(device=DEVICE)
        tabpfn_model.fit(X_train_tabpfn.values, y_train_tabpfn.values)

        tabpfn_probs = tabpfn_model.predict_proba(X_test_final.values)[:, 1]
        tabpfn_preds = (tabpfn_probs >= 0.5).astype(int)

        tabpfn_results = {
            "model": "TabPFN (foundation model)",
            "f1": f1_score(y_test_final, tabpfn_preds),
            "roc_auc": roc_auc_score(y_test_final, tabpfn_probs),
        }
        print("TabPFN F1:", round(tabpfn_results["f1"], 4),
              "| ROC-AUC:", round(tabpfn_results["roc_auc"], 4))
    except Exception as exc:
        print("TabPFN fit or predict failed; skipping the foundation model benchmark. Error:", exc)
        tabpfn_results = None
else:
    print("TabPFN is unavailable in this environment; the comparison in Section 13.3 proceeds "
          "with the classical model and the feedforward network only.")

### **13.3 - Three-Way Model Comparison**

The chart below extends the two-model comparison from Section 12.2 with TabPFN's score, when
available. TabPFN never ran a training loop against this dataset and never saw the feature engineering
choices tuned to it; it conditioned on 3,000 labeled rows at inference time and produced predictions
directly. Landing near the classical model and the feedforward network on F1, despite that handicap, is
the finding that matters here: a model pretrained once on synthetic tasks unrelated to network security
substitutes for a substantial share of the per-dataset training effort the other two models required.

In [ ]:
model_comparison_full = model_comparison.copy()
if tabpfn_results is not None:
    model_comparison_full = pd.concat([
        model_comparison_full,
        pd.DataFrame([{"model": tabpfn_results["model"], "f1": tabpfn_results["f1"],
                        "roc_auc": tabpfn_results["roc_auc"]}]),
    ], ignore_index=True)

model_comparison_full

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=model_comparison_full, x="model", y="f1", ax=ax, color="slateblue")
ax.set_title("F1 Score by Model, Including the Foundation Model Benchmark")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "model_comparison_with_tabpfn.png"), dpi=100)
plt.show()

if tabpfn_results is None:
    print("TabPFN was unavailable in this run, so this chart reflects the classical model and the "
          "feedforward network only.")

## **14 - Explainability: SHAP**

### **14.1 - Why SHAP for This Case**

Permutation importance and partial dependence, used elsewhere in this case study series, describe a
model's behavior in aggregate: which features matter on average and how the prediction responds as one
feature sweeps across its range. This case study has a different priority. A security analyst reviewing
a single flagged flow needs to know why that specific flow was flagged, which features pushed its
predicted probability toward malicious and by how much, so that the alert can be acted on rather than
taken on faith. That is a local, per-prediction attribution problem, and it matters more here than in a
typical regression case because malicious flows are a minority class: an aggregate importance ranking
can be dominated by the majority-class behavior of benign flows and obscure what specifically separates
an individual attack flow from the benign background it is embedded in. SHAP's TreeExplainer computes
exact per-prediction feature attributions for tree ensembles efficiently and additively (each flow's
attributions sum to its predicted log-odds relative to a baseline), which makes it well suited to both
the global ranking this notebook needs for the law rediscovery in Section 15 and the local, per-flow
explanation the agentic layer in Section 16 needs to reason over.

### **14.2 - Global Feature Importance**

SHAP values are computed on the reference classical model against the held-out test set. The mean
absolute SHAP value per feature gives a global importance ranking, directly comparable in spirit to the
permutation importance ranking used elsewhere in this series, but computed from exact per-prediction
attributions rather than repeated shuffling.

In [ ]:
explainer = shap.TreeExplainer(classical_model)
shap_values = explainer.shap_values(X_test_final)

# GradientBoostingClassifier and RandomForestClassifier return SHAP values in slightly different
# shapes; normalize to a single 2D array of attributions for the positive (malicious) class.
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
elif shap_values.ndim == 3:
    shap_values_pos = shap_values[:, :, 1]
else:
    shap_values_pos = shap_values

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": FULL_FEATURES,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_importance_df.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_values_pos, X_test_final, plot_type="bar", show=False, max_display=15)
plt.title("SHAP Global Feature Importance")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_global_importance.png"), dpi=100)
plt.show()

In [ ]:
fig = plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values_pos, X_test_final, show=False, max_display=15)
plt.title("SHAP Value Distribution by Feature")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_beeswarm.png"), dpi=100)
plt.show()

### **14.3 - Local Explanation for a Flagged Flow**

A per-flow waterfall plot shows exactly which features drove one specific test-set flow's
prediction toward malicious, the level of detail a security analyst reviewing an individual alert
actually needs. The flow chosen below is a true positive with high predicted probability, so the
attribution reflects a flow the model is confident is an attack.

In [ ]:
test_probs = classical_model.predict_proba(X_test_final)[:, 1]
candidate_idx = np.where((y_test_final.values == 1) & (test_probs > 0.9))[0]
flagged_row_pos = candidate_idx[0] if len(candidate_idx) > 0 else np.argmax(test_probs)

flagged_flow_id = original_test.iloc[flagged_row_pos]["flow_id"]
flagged_attack_type = original_test.iloc[flagged_row_pos]["attack_type"]
print("Flagged flow:", flagged_flow_id, "| true attack type:", flagged_attack_type,
      "| predicted probability malicious:", round(float(test_probs[flagged_row_pos]), 4))

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = np.atleast_1d(base_value)
    base_value = base_value[1] if len(base_value) > 1 else base_value[0]

explanation = shap.Explanation(
    values=shap_values_pos[flagged_row_pos],
    base_values=base_value,
    data=X_test_final.iloc[flagged_row_pos].values,
    feature_names=FULL_FEATURES,
)
shap.plots.waterfall(explanation, show=False, max_display=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "shap_waterfall_flagged_flow.png"), dpi=100)
plt.show()

## **15 - The Law Rediscovery Moment: Statistical Deviation Over Signatures**

### **15.1 - What the Explanation Surfaces**

The SHAP global importance ranking in Section 14.2 places a small handful of features at the top:
packet-rate deviation from baseline, byte-to-packet ratio (a direct measure of how thin a flow's
payload is relative to its packet count), destination-port-count deviation from baseline, raw
destination-port fan-out, and connection repetition. These features were mixed in among eleven
human-engineered features, six AI-suggested features, and sixteen raw telemetry columns, thirty-three
candidates in total, and none of them was labeled as the answer at any point in this notebook. The
model was free to rely on raw packet counts, raw flag counts, or any AI-suggested transform instead,
and largely did not. Byte-rate deviation and SYN ratio carry the same qualitative signal as the
top-ranked features, but overlap with them enough (byte-rate deviation with packet-rate deviation,
SYN ratio with the raw flag counts and byte-to-packet ratio) that the tree ensemble routes its
attribution through one representative of each correlated cluster rather than splitting credit evenly,
a known property of SHAP on tree ensembles with correlated inputs, not evidence that the overlapping
feature carries no signal.

### **15.2 - A Deviation Score Separates All Three Attack Types**

Denning's model does not require a single scalar anomaly score. It requires monitoring several
behavioral measures against each subject's baseline and flagging the subject when any one of them
deviates enough. As a post-hoc check, a composite anomaly score is built for the first time in this
notebook, purely as a diagnostic, by standardizing four deviation and repetition features SHAP ranked
highest and taking the maximum standardized value per flow, rather than the average, since each of the
three attack types deviates on a different axis (packet and byte rate for SYN floods, port-count
fan-out for port scans, connection repetition for brute-force campaigns) and no single attack type
deviates strongly on all of them at once. The model was never given this composite score as a feature;
it only ever saw the four ingredient features separately, alongside everything else.

In [ ]:
diagnostic = original_test.copy()
diagnostic["predicted_prob_malicious"] = test_probs

deviation_cols = ["pkt_rate_deviation", "byte_rate_deviation", "port_count_deviation",
                   "connection_repeat_ratio"]
deviation_z = (diagnostic[deviation_cols] - diagnostic[deviation_cols].mean()) / diagnostic[deviation_cols].std()
diagnostic["anomaly_score"] = deviation_z.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=diagnostic, x="attack_type", y="anomaly_score", ax=axes[0])
axes[0].set_title("Composite Anomaly Score by True Traffic Class")
axes[0].tick_params(axis="x", rotation=20)

sns.scatterplot(data=diagnostic, x="anomaly_score", y="predicted_prob_malicious", hue="attack_type",
                 alpha=0.5, s=18, ax=axes[1])
axes[1].set_title("Predicted Malicious Probability vs. Composite Anomaly Score")
axes[1].set_xlabel("Composite anomaly score (never seen by the model)")
axes[1].set_ylabel("Predicted probability malicious")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "law_rediscovery.png"), dpi=100)
plt.show()

print(diagnostic.groupby("attack_type")["anomaly_score"].median().round(3))

### **15.3 - The Historical Payoff**

The separation recovered above is the same finding Denning formalized in 1987: monitoring how far
current behavior deviates from an established baseline, across several behavioral measures at once,
separates normal activity from intrusions, and it does so across attack types that look nothing alike
at the packet level. A SYN flood, a port scan, and a brute-force campaign share almost no surface-level
signature; one is defined by an incomplete handshake, one by destination-port fan-out, one by repeated
connections at regular intervals. What they share is that each pushes at least one of a source's
behavioral measures far from where that source's own history says it should be, though rarely the same
measure twice. A signature-based system built before any of these three attack types existed would have
caught none of them. A statistical deviation model, built on Denning's 1987 framing and never told which
deviation mattered, catches all three, because it was never
looking for a signature in the first place.

## **16 - Agentic Layer: LangGraph Security Recommendation**

### **16.1 - Overview**

This section builds a small LangGraph graph with two nodes. The first node, `diagnose`, is a
deterministic function that takes a flagged flow's predicted probability and its top SHAP-contributing
features (from Section 14.3) and turns them into a structured, plain-language diagnosis: which attack
signature the flow resembles and which specific features drove that call. The second node,
`recommend`, sends that diagnosis to the Hugging Face router and asks the language model to draft a
short security-analyst recommendation, such as rate-limiting a source IP for a suspected SYN flood or
disabling password-based login for a suspected brute-force campaign. The graph is deliberately small: a
diagnose step and a recommend step are enough to demonstrate an agentic layer that turns a numeric
model output into an actionable note, without adding nodes that do not change the outcome.

In [ ]:
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, END

class FlowState(TypedDict):
    flow_id: str
    predicted_prob: float
    top_features: List[str]
    syn_ratio: float
    unique_dst_ports_window: float
    connection_repeat_ratio: float
    diagnosis: Optional[str]
    recommendation: Optional[str]

def diagnose_node(state: FlowState) -> FlowState:
    prob = state["predicted_prob"]
    top_feats = ", ".join(state["top_features"])

    if state["syn_ratio"] > 0.85 and state["unique_dst_ports_window"] <= 2:
        pattern = "SYN-flood denial-of-service"
    elif state["unique_dst_ports_window"] > 20:
        pattern = "port scan / reconnaissance"
    elif state["connection_repeat_ratio"] > 5:
        pattern = "brute-force login attempt"
    else:
        pattern = "an unclassified statistical anomaly"

    diagnosis = (
        f"Flow {state['flow_id']} was flagged with predicted malicious probability {prob:.2f}. "
        f"The traffic pattern most resembles {pattern}. The top SHAP-contributing features for this "
        f"flow were: {top_feats}. SYN ratio is {state['syn_ratio']:.2f}, destination-port fan-out is "
        f"{state['unique_dst_ports_window']:.0f}, and connection repeat ratio is "
        f"{state['connection_repeat_ratio']:.2f}."
    )
    state["diagnosis"] = diagnosis
    return state

def recommend_node(state: FlowState) -> FlowState:
    prompt = f"""You are a network security analyst assistant. Given this diagnosis of a flagged
network flow, write a 3 to 4 sentence recommendation in plain language for the on-call security
analyst. Recommend a specific, concrete containment or mitigation action appropriate to the traffic
pattern described (for example: rate-limit or block the source IP, disable password-based
authentication in favor of key-based login, or throttle connections per source), and note the
confidence implied by the predicted probability.

Diagnosis: {state['diagnosis']}
"""
    llm_text = call_hf_llm(prompt, max_tokens=250, temperature=0.5)
    if llm_text is None:
        llm_text = (
            "HF router unavailable, using fallback recommendation. Based on the diagnosis, rate-limit "
            "or temporarily block the source IP at the network boundary, and if the pattern indicates "
            "repeated login attempts against a single service, disable password-based authentication "
            "on that service in favor of key-based or multi-factor login until the source is cleared."
        )
    state["recommendation"] = llm_text
    return state

graph = StateGraph(FlowState)
graph.add_node("diagnose", diagnose_node)
graph.add_node("recommend", recommend_node)
graph.set_entry_point("diagnose")
graph.add_edge("diagnose", "recommend")
graph.add_edge("recommend", END)
intrusion_agent = graph.compile()

print("LangGraph agent compiled with nodes: diagnose -> recommend")

### **16.2 - Running the Agent on the Flagged Flow**

The graph is invoked on the same flagged test-set flow used for the SHAP waterfall plot in Section
14.3, using that flow's own predicted probability and top SHAP-contributing features as the numeric
input the agent reasons over.

In [ ]:
top_feature_names = shap_importance_df.head(5)["feature"].tolist()

sample_row = original_test.iloc[flagged_row_pos]
sample_state: FlowState = {
    "flow_id": sample_row["flow_id"],
    "predicted_prob": float(test_probs[flagged_row_pos]),
    "top_features": top_feature_names,
    "syn_ratio": float(sample_row["syn_ratio"]) if "syn_ratio" in sample_row else float(
        flows_feat.loc[sample_row.name, "syn_ratio"]),
    "unique_dst_ports_window": float(sample_row["unique_dst_ports_window"]),
    "connection_repeat_ratio": float(flows_feat.loc[sample_row.name, "connection_repeat_ratio"]),
    "diagnosis": None,
    "recommendation": None,
}

result_state = intrusion_agent.invoke(sample_state)
print("Diagnosis:\n", result_state["diagnosis"])
print("\nRecommendation:\n", result_state["recommendation"])

### **16.3 - From Fixed Pipeline to Autonomous Agent**

The fixed pipeline in Sections 16.1 and 16.2 has scripted control flow: `diagnose` always runs,
then `recommend` always runs, in that order, regardless of what either node produces. It never calls a
tool; both nodes compute or generate text directly from the state they are handed. It carries no memory
between invocations; each call to `intrusion_agent.invoke` starts from a blank state and knows nothing
about any flow scored before it. Three axes distinguish a scripted agentic pipeline from an autonomous
agent: planning (who decides what step runs next), tool use (whether the model can call functions rather
than only read numbers already computed for it), and memory (whether past sessions inform the current
one). The fixed pipeline has none of the three: its two-node order is hardcoded in Python, its nodes
never call a tool, and it never reads or writes state outside a single invocation.

The agent built in this subsection has all three. A LangGraph conditional edge lets the language model
decide, turn by turn, which tool to call next and when it has gathered enough information to stop,
rather than following a Python if/else chain written in advance. Four tools give it real actions to
take: computing a flow's derived statistics itself rather than trusting numbers handed to it, querying
the trained model, recalling similar past incidents from an append-only log, and escalating a flow to
the SOC (security operations center) as a terminal action. That log is the memory: every session the
agent runs appends the flow it examined, its diagnosis, and its recommendation to a JSON Lines file, so
a later session investigating a similar flow can retrieve precedent instead of treating every flow as
first contact.

### **16.4 - Building the ReAct Tool-Calling Agent**

Tool-calling reliability matters more here than in Sections 9 and 10's single-shot GenAI calls,
since the agent must recognize when to call a tool, parse its own previous tool result, and decide
whether to call another tool or stop. The 1.5B-parameter model used for the fixed pipeline's
`recommend` node is small enough that its tool-calling can be inconsistent, so this agent uses a larger
free-tier Qwen2.5 variant, `Qwen/Qwen2.5-7B-Instruct`, through the same Hugging Face router client,
reserving the smaller model for the fixed pipeline's simpler single-shot generation task. Four Python
functions are exposed to the model as callable tools, described with the same OpenAI-compatible tool
schema the router client already accepts: `compute_flow_stats` derives a flow's rate and deviation
statistics, `predict_flow_label` calls the trained classical model, `recall_similar_incidents` searches
the memory log for similar past flows, and `flag_for_soc_escalation` is the terminal action that closes
out an investigation. The `agent` node calls the language model with these tools bound and enabled; a
conditional edge routes to a `tools` node whenever the response includes tool calls and to `END` once
the model responds with none, looping back to `agent` after every tool call so the model can read the
result before deciding what to do next.

In [ ]:
from typing import Dict, Any

AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
AGENT_MEMORY_PATH = os.path.join(DATA_DIR, "agent_memory.jsonl")
MEMORY_FEATURE_COLS = ["syn_ratio", "pkt_rate_deviation", "byte_rate_deviation",
                       "port_count_deviation", "connection_repeat_ratio"]

_memory_feature_means = original_train[MEMORY_FEATURE_COLS].mean()
_memory_feature_stds = original_train[MEMORY_FEATURE_COLS].std().replace(0, 1.0)


def _lookup_test_flow(flow_id: str):
    match = original_test[original_test["flow_id"] == flow_id]
    return None if match.empty else match.iloc[0]


def compute_flow_stats(flow_id: str) -> Dict[str, Any]:
    """Derive rate, ratio, and baseline-deviation statistics for a flow in the scored test set."""
    row = _lookup_test_flow(flow_id)
    if row is None:
        return {"error": f"flow_id {flow_id} not found in the test set"}
    cols = MEMORY_FEATURE_COLS + ["unique_dst_ports_window"]
    return {col: round(float(row[col]), 4) for col in cols}


def predict_flow_label(flow_id: str) -> Dict[str, Any]:
    """Return the reference classical model's predicted probability and label for a test-set flow."""
    match = original_test[original_test["flow_id"] == flow_id]
    if match.empty:
        return {"error": f"flow_id {flow_id} not found in the test set"}
    pos = original_test.index.get_loc(match.index[0])
    prob = float(test_probs[pos])
    return {"flow_id": flow_id, "predicted_prob_malicious": round(prob, 4),
            "predicted_label": "malicious" if prob >= 0.5 else "benign"}


def recall_similar_incidents(flow_id: str, n: int = 3) -> Dict[str, Any]:
    """Return the n most similar past logged incidents to this flow, by normalized feature distance."""
    row = _lookup_test_flow(flow_id)
    if row is None:
        return {"error": f"flow_id {flow_id} not found in the test set"}
    if not os.path.exists(AGENT_MEMORY_PATH):
        return {"matches": [], "note": "memory log is empty; no past sessions recorded yet"}

    target_vec = (row[MEMORY_FEATURE_COLS] - _memory_feature_means) / _memory_feature_stds

    logged = []
    with open(AGENT_MEMORY_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                logged.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    scored = []
    for entry in logged:
        try:
            entry_vec = pd.Series({col: entry["features"][col] for col in MEMORY_FEATURE_COLS})
            entry_vec = (entry_vec - _memory_feature_means) / _memory_feature_stds
            distance = float(np.sqrt(((target_vec - entry_vec) ** 2).sum()))
            scored.append((distance, entry))
        except (KeyError, TypeError):
            continue

    scored.sort(key=lambda pair: pair[0])
    matches = [
        {"distance": round(dist, 3), "flow_id": entry["flow_id"],
         "predicted_label": entry["predicted_label"], "diagnosis": entry["diagnosis"]}
        for dist, entry in scored[:n]
    ]
    return {"matches": matches}


def flag_for_soc_escalation(flow_id: str, reason: str) -> Dict[str, Any]:
    """Terminal action: mark a flow as escalated to the SOC (security operations center)."""
    return {"status": "escalated", "flow_id": flow_id, "reason": reason}


AGENT_TOOL_FUNCTIONS = {
    "compute_flow_stats": compute_flow_stats,
    "predict_flow_label": predict_flow_label,
    "recall_similar_incidents": recall_similar_incidents,
    "flag_for_soc_escalation": flag_for_soc_escalation,
}

AGENT_TOOLS = [
    {"type": "function", "function": {
        "name": "compute_flow_stats",
        "description": "Derive rate, ratio, and baseline-deviation statistics for a flagged flow.",
        "parameters": {
            "type": "object",
            "properties": {"flow_id": {"type": "string", "description": "The flow identifier to inspect."}},
            "required": ["flow_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "predict_flow_label",
        "description": "Get the trained model's predicted probability and label for a flow.",
        "parameters": {
            "type": "object",
            "properties": {"flow_id": {"type": "string", "description": "The flow identifier to score."}},
            "required": ["flow_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "recall_similar_incidents",
        "description": "Search past logged agent sessions for the most similar prior flows.",
        "parameters": {
            "type": "object",
            "properties": {
                "flow_id": {"type": "string", "description": "The flow identifier to find precedent for."},
                "n": {"type": "integer", "description": "Number of similar past incidents to return."},
            },
            "required": ["flow_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "flag_for_soc_escalation",
        "description": "Escalate a flow to the SOC. Call this only once escalation is warranted.",
        "parameters": {
            "type": "object",
            "properties": {
                "flow_id": {"type": "string", "description": "The flow identifier being escalated."},
                "reason": {"type": "string", "description": "Short justification for escalating."},
            },
            "required": ["flow_id", "reason"],
        },
    }},
]


class AgentState(TypedDict):
    flow_id: str
    messages: List[dict]


def agent_node(state: AgentState) -> AgentState:
    if hf_client is None:
        state["messages"].append({"role": "assistant",
                                   "content": "No HF_TOKEN configured; cannot run the tool-calling agent."})
        return state
    try:
        response = hf_client.chat.completions.create(
            model=AGENT_MODEL,
            messages=state["messages"],
            tools=AGENT_TOOLS,
            tool_choice="auto",
            max_tokens=600,
            temperature=0.2,
        )
        msg = response.choices[0].message
        assistant_message = {"role": "assistant", "content": msg.content or ""}
        if getattr(msg, "tool_calls", None):
            assistant_message["tool_calls"] = [
                {"id": call.id, "type": "function",
                 "function": {"name": call.function.name, "arguments": call.function.arguments}}
                for call in msg.tool_calls
            ]
        state["messages"].append(assistant_message)
    except Exception as exc:
        print("Autonomous agent LLM call failed, ending the turn with a fallback message. Error:", exc)
        state["messages"].append({"role": "assistant",
                                   "content": "LLM call failed; defer to the fixed pipeline's diagnosis instead."})
    return state


def tools_node(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]
    for call in last_message.get("tool_calls", []):
        name = call["function"]["name"]
        try:
            args = json.loads(call["function"]["arguments"])
        except (json.JSONDecodeError, TypeError):
            args = {}
        func = AGENT_TOOL_FUNCTIONS.get(name)
        try:
            result = func(**args) if func is not None else {"error": f"unknown tool {name}"}
        except Exception as exc:
            result = {"error": str(exc)}
        state["messages"].append({"role": "tool", "tool_call_id": call["id"], "content": json.dumps(result)})
    return state


def route_after_agent(state: AgentState) -> str:
    if state["messages"][-1].get("tool_calls"):
        return "tools"
    return END


agent_graph = StateGraph(AgentState)
agent_graph.add_node("agent", agent_node)
agent_graph.add_node("tools", tools_node)
agent_graph.set_entry_point("agent")
agent_graph.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})
agent_graph.add_edge("tools", "agent")
autonomous_security_agent = agent_graph.compile()

print("Autonomous agent compiled with a conditional agent -> tools loop, tool-calling model:", AGENT_MODEL)

### **16.5 - Running the Autonomous Agent on the Flagged Flow**

The agent below investigates the same flagged flow used throughout Sections 14.3 and 16.2, given
only its flow_id and an instruction to decide whether escalation is warranted. It is free to call any
tool, in any order, and to stop once it judges it has enough information; the trace below prints every
tool call and every intermediate message so the reasoning stays visible rather than only the final
answer. A successful run is logged to the memory file described in Section 16.3, so a later invocation
of `recall_similar_incidents` can find it.

In [ ]:
def log_agent_session(flow_id, features, predicted_label, diagnosis, recommendation):
    os.makedirs(DATA_DIR, exist_ok=True)
    entry = {
        "flow_id": flow_id,
        "features": features,
        "predicted_label": predicted_label,
        "diagnosis": diagnosis,
        "recommendation": recommendation,
    }
    with open(AGENT_MEMORY_PATH, "a") as f:
        f.write(json.dumps(entry) + "\n")


def run_autonomous_agent(flow_id):
    system_prompt = (
        "You are an autonomous SOC (security operations center) analyst agent for a network intrusion "
        "detection system. You can call compute_flow_stats, predict_flow_label, and "
        "recall_similar_incidents to investigate a flagged flow, and flag_for_soc_escalation once you "
        "have decided the flow should be escalated. Investigate using whichever tools you judge useful, "
        "in whatever order you judge useful, and stop once you have enough information. Only call "
        "flag_for_soc_escalation if escalation is warranted, with a short reason."
    )
    if hf_client is None:
        print("No HF_TOKEN configured; the autonomous agent needs a live tool-calling model and cannot "
              "run. The fixed pipeline result from Section 16.2 stands in its place.")
        return None

    initial_state: AgentState = {
        "flow_id": flow_id,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Investigate flow {flow_id} and decide what action to take."},
        ],
    }
    try:
        result = autonomous_security_agent.invoke(initial_state, {"recursion_limit": 12})
    except Exception as exc:
        print("Autonomous agent run failed. Error:", exc)
        return None

    for message in result["messages"]:
        if message["role"] == "tool":
            print(f"  [tool result] {message['content']}")
        elif message["role"] == "assistant" and message.get("tool_calls"):
            calls = ", ".join(f"{c['function']['name']}({c['function']['arguments']})"
                               for c in message["tool_calls"])
            print(f"[agent] calling: {calls}")
        elif message["role"] == "assistant" and message["content"]:
            print(f"[agent] {message['content']}")

    return result


agent_run_result = run_autonomous_agent(flagged_flow_id)

if agent_run_result is not None:
    final_text = next(
        (m["content"] for m in reversed(agent_run_result["messages"])
         if m["role"] == "assistant" and m["content"]),
        "",
    )
    flow_features = compute_flow_stats(flagged_flow_id)
    log_agent_session(
        flow_id=flagged_flow_id,
        features=flow_features,
        predicted_label=predict_flow_label(flagged_flow_id).get("predicted_label"),
        diagnosis=result_state["diagnosis"],
        recommendation=final_text,
    )
    print("\nSession logged to", AGENT_MEMORY_PATH)

## **17 - Interactive Prediction Demo**

### **17.1 - Overview**

The function below ties the full pipeline together for a single candidate flow. It accepts the raw
telemetry values a monitoring appliance would report, derives the same human-engineered and
AI-suggested features used in training, predicts the malicious probability with the reference
classical model, and runs the LangGraph agent to produce a security recommendation.

In [ ]:
def predict_flow(flow_duration_s, packet_count, byte_count, syn_count, ack_count, fin_count,
                  rst_count, dst_port, unique_dst_ports_window, src_port_entropy_window,
                  flows_per_src_dst_pair_window, inter_arrival_mean_ms, inter_arrival_cv,
                  baseline_pkt_rate_src, baseline_byte_rate_src, baseline_unique_ports_src):
    row = pd.DataFrame([{
        "flow_duration_s": flow_duration_s,
        "packet_count": packet_count,
        "byte_count": byte_count,
        "syn_count": syn_count,
        "ack_count": ack_count,
        "fin_count": fin_count,
        "rst_count": rst_count,
        "dst_port": dst_port,
        "unique_dst_ports_window": unique_dst_ports_window,
        "src_port_entropy_window": src_port_entropy_window,
        "flows_per_src_dst_pair_window": flows_per_src_dst_pair_window,
        "inter_arrival_mean_ms": inter_arrival_mean_ms,
        "inter_arrival_cv": inter_arrival_cv,
        "baseline_pkt_rate_src": baseline_pkt_rate_src,
        "baseline_byte_rate_src": baseline_byte_rate_src,
        "baseline_unique_ports_src": baseline_unique_ports_src,
    }])

    row = add_human_engineered_features(row)
    row, _ = build_ai_features(row, ai_feature_specs)

    predicted_prob = float(classical_model.predict_proba(row[FULL_FEATURES])[:, 1][0])
    predicted_label = "malicious" if predicted_prob >= 0.5 else "benign"

    row_shap = explainer.shap_values(row[FULL_FEATURES])
    if isinstance(row_shap, list):
        row_shap_pos = row_shap[1][0]
    elif np.ndim(row_shap) == 3:
        row_shap_pos = row_shap[0, :, 1]
    else:
        row_shap_pos = row_shap[0]
    top_features_local = pd.Series(np.abs(row_shap_pos), index=FULL_FEATURES).sort_values(
        ascending=False).head(5).index.tolist()

    state: FlowState = {
        "flow_id": "USER-INPUT-FLOW",
        "predicted_prob": predicted_prob,
        "top_features": top_features_local,
        "syn_ratio": float(row["syn_ratio"].iloc[0]),
        "unique_dst_ports_window": float(unique_dst_ports_window),
        "connection_repeat_ratio": float(row["connection_repeat_ratio"].iloc[0]),
        "diagnosis": None,
        "recommendation": None,
    }
    result = intrusion_agent.invoke(state)

    print(f"Predicted label: {predicted_label} (probability malicious: {predicted_prob:.3f})")
    print(f"Top contributing features: {top_features_local}")
    print(f"\nDiagnosis: {result['diagnosis']}")
    print(f"\nRecommendation: {result['recommendation']}")
    return result

_ = predict_flow(
    flow_duration_s=0.08, packet_count=1400, byte_count=84000, syn_count=1360, ack_count=40,
    fin_count=0, rst_count=2, dst_port=443, unique_dst_ports_window=1, src_port_entropy_window=3.1,
    flows_per_src_dst_pair_window=5, inter_arrival_mean_ms=1.2, inter_arrival_cv=0.3,
    baseline_pkt_rate_src=8.0, baseline_byte_rate_src=5200.0, baseline_unique_ports_src=2.0,
)

In [ ]:
# A second example, a benign large file transfer that should not be flagged despite its size.
_ = predict_flow(
    flow_duration_s=45.0, packet_count=9000, byte_count=6_500_000, syn_count=1, ack_count=8997,
    fin_count=2, rst_count=0, dst_port=443, unique_dst_ports_window=2, src_port_entropy_window=3.8,
    flows_per_src_dst_pair_window=1, inter_arrival_mean_ms=90.0, inter_arrival_cv=0.9,
    baseline_pkt_rate_src=190.0, baseline_byte_rate_src=140000.0, baseline_unique_ports_src=2.0,
)

### **17.2 - Calling the Autonomous Agent on a Demo Flow**

The autonomous agent's tools look up a flow by flow_id in the scored test set, rather than
accepting arbitrary user-supplied telemetry directly, so this demo runs it against a second test-set
flow instead of the free-form inputs above. A port-scan flow is chosen deliberately, since it lets
`recall_similar_incidents` check for precedent against the SYN-flood-pattern flow logged in Section
16.5, a different attack type, exercising the tool against a real memory log rather than an empty one.

In [ ]:
port_scan_test_flows = original_test[original_test["attack_type"] == "port_scan"]
demo_flow_id = port_scan_test_flows.iloc[0]["flow_id"] if len(port_scan_test_flows) > 0 else flagged_flow_id

print("Running the autonomous agent on flow:", demo_flow_id)
_ = run_autonomous_agent(demo_flow_id)

## **18 - Conclusion and Takeaways**

### **18.1 - Conclusion**

This notebook built a network intrusion classification pipeline from a synthetic dataset grounded
in the statistical signatures of three attack types and one benign traffic model, rather than from a
black-box data source. The pipeline combined hand-engineered security-analyst ratios and
deviation-from-baseline features with LLM-suggested feature transforms, showed that the combined
feature set outperforms either source alone, and used an LLM-guided augmentation strategy that filled
in under-represented regions of the traffic space while a fixed original holdout confirmed
generalization stayed within tolerance. A gradient boosting model and a compact PyTorch network both
reached a strong fit on held-out flows despite the class imbalance, and SHAP, chosen over permutation
importance and partial dependence for its per-flow local attributions, recovered statistical deviation
from a source's established baseline, alongside destination-port fan-out and connection repetition, as
the dominant drivers of a malicious classification. A composite anomaly score built after the fact from
nothing but those deviation and repetition features, taking the maximum deviation across several
behavioral measures rather than their average, separated all three attack types from benign traffic at
once, reproducing Dorothy Denning's 1987 finding that intrusions are detectable as statistical
deviations from established behavior rather than as instances of known attack signatures. A pretrained tabular foundation model, TabPFN, was benchmarked against both trained models on the same held-out flows, trained on a stratified subsample of the training set rather than the full training data, and landed close to their scores without running a training loop of its own. The agentic layer was built twice: once as a fixed two-node pipeline with no tool use and no memory across invocations, and once as a ReAct-style LangGraph agent that plans its own investigation, calls tools to compute statistics, query the model, and recall past incidents from an append-only log, and decides for itself when to escalate a flow to the SOC (security operations center).

### **18.2 - Takeaways**

- Three attack types with almost no shared surface-level signature, a SYN flood, a port scan, and a
  brute-force campaign, are all detectable through the same underlying property: each pushes a source's
  behavior far from its own established baseline. A model given the raw ingredients separately can
  recover this without being told which deviation matters.
- Denning's 1987 statistical anomaly model predates every attack type in this dataset by years. A
  model trained in 2026 on synthetic flow telemetry rediscovering the same principle through SHAP is
  evidence for how durable that principle is across four decades of attack evolution.
- Benign flow sizes and durations follow the heavy-tailed, self-similar pattern Leland et al. reported
  in real Ethernet traffic in 1994: a small share of flows carries most of the bytes. Any monitoring
  threshold based on absolute size alone risks flagging legitimate bulk transfers, which is why
  deviation from a per-source baseline holds up as a signal where an absolute size threshold does not.
- LLM-suggested features and hand-engineered security features are complementary rather than redundant,
  and an ablation study is the correct way to check that before committing to either source alone.
- SHAP's per-flow local attributions serve a downstream use case that aggregate rankings do not: an
  analyst reviewing one flagged alert needs to know why that specific flow was flagged. That local
  attribution is why this case study chose SHAP over the permutation importance and partial dependence
  approach used elsewhere in this series.
- A pretrained tabular foundation model is a distinct capability tier from a custom-trained model: TabPFN never ran a training loop against this dataset, conditioned on a stratified subsample of the training rows at inference time, and still landed close to the classical model and the feedforward network on F1. Subsampling to stay inside TabPFN's documented operating range is a tradeoff worth stating explicitly rather than a detail to gloss over.
- A scripted agentic pipeline and an autonomous agent differ on three axes: planning, tool use, and memory. The fixed LangGraph pipeline in Section 16.1 has none of the three; the ReAct-style agent in Section 16.4 has all three, deciding which tool to call and when to stop, computing its own statistics and predictions through tool calls rather than trusting precomputed numbers, and reading and writing an append-only memory log so later sessions can reference precedent instead of starting from a blank state.